# Clasificación Multi-Clase con Sentence Transformers
Pipeline completo: carga de corpus → embeddings semánticos → Logistic Regression (OvR) → métricas por clase

## ¿Qué cambia respecto a TF-IDF?

| Elemento | TF-IDF (sesion_4) | SentenceTransformer (sesion_5) |
|---|---|---|
| Representación | Frecuencias de términos (sparse 1000-dim) | Embeddings semánticos (dense **384-dim**) |
| Vocabulario | Limitado al corpus de entrenamiento | Generaliza a palabras **nunca vistas** |
| `fit()` necesario | Sí — `fit_transform()` | **No** — modelo pre-entrenado |
| Multilingüe | No | **Sí** — `paraphrase-multilingual-MiniLM-L12-v2` |
| Sensibilidad semántica | Baja (no entiende sinónimos) | **Alta** — captura significado |
| Estrategia del modelo | OvR | OvR — **sin cambios** |
| Número de clases | 5 | 5 — **sin cambios** |

> **SentenceTransformer:** convierte cada texto en un vector denso de 384 dimensiones  
> donde textos con significado similar quedan geométricamente cerca.  
> El clasificador aprende fronteras de decisión sobre esos vectores.

In [ ]:
# Instalar librería (solo necesario en Colab)
!pip install sentence-transformers -q

## 0 · Subir el corpus

In [ ]:
# subir corpus_sentimiento_reviews.json desde tu computadora
from google.colab import files
uploaded = files.upload()

## 1 · Imports y configuración

In [ ]:
import json, io
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

CLASES       = ['peliculas', 'restaurantes', 'productos', 'servicios', 'hoteles']
RANDOM_STATE = 42

## 2 · Carga del corpus

La única diferencia con el notebook de sentimiento:  
usamos `item['categoria']` como etiqueta en lugar de `item['sentimiento']`.

In [ ]:
data = json.load(io.BytesIO(uploaded['corpus_sentimiento_reviews.json']))

reviews    = [item['texto']     for item in data['reviews']]
categorias = [item['categoria'] for item in data['reviews']]

print(f'Total de reseñas: {len(reviews)}')
print()
print('Ejemplos por categoría:')
for c in CLASES:
    print(f'  {c:<15}: {categorias.count(c):>3}')

## 3 · Métricas desde cero (sin sklearn)

Las **mismas funciones** del notebook de sentimiento, sin modificar.  
`pos_label` acepta cualquier string, así que funcionan con las 5 categorías.

In [ ]:
def confusion_matrix_manual(y_true, y_pred, pos_label='positivo'):
    TP = sum(t == pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    TN = sum(t != pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    FP = sum(t != pos_label and p == pos_label for t, p in zip(y_true, y_pred))
    FN = sum(t == pos_label and p != pos_label for t, p in zip(y_true, y_pred))
    return TP, TN, FP, FN

def accuracy(y_true, y_pred):
    TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
    return (TP + TN) / (TP + TN + FP + FN)

def precision(y_true, y_pred, pos_label='positivo'):
    TP, _, FP, _ = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FP) if (TP + FP) > 0 else 0.0

def recall(y_true, y_pred, pos_label='positivo'):
    TP, _, _, FN = confusion_matrix_manual(y_true, y_pred, pos_label)
    return TP / (TP + FN) if (TP + FN) > 0 else 0.0

def f1(y_true, y_pred, pos_label='positivo'):
    p = precision(y_true, y_pred, pos_label)
    r = recall(y_true, y_pred, pos_label)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

## 4 · Matriz de confusión N×N

Para 5 clases la matriz ya no es 2×2.  
Cada fila es la clase **real**, cada columna la clase **predicha**.  
Los valores de la diagonal principal son las predicciones correctas.

```
              predicho →
real ↓   peliculas  restaurantes  productos  servicios  hoteles
peliculas     TP            FP        FP         FP        FP
restaurantes  FP            TP        FP         FP        FP
...           ...           ...       ...        ...       ...
```

In [ ]:
def confusion_matrix_multiclase(y_true, y_pred, clases):
    """Imprime la matriz de confusión N×N en formato ASCII."""
    ancho = 14
    # encabezado
    cabecera = ' ' * ancho + ''.join(c[:ancho].ljust(ancho) for c in clases)
    print(cabecera)
    print('-' * (ancho * (len(clases) + 1)))
    # filas
    for real in clases:
        fila = real[:ancho].ljust(ancho)
        for pred in clases:
            count = sum(t == real and p == pred for t, p in zip(y_true, y_pred))
            fila += str(count).ljust(ancho)
        print(fila)

## 5 · Split Train / Test

Misma configuración que el notebook de sentimiento.  
`stratify=categorias` garantiza que cada clase tenga la misma proporción en train y test.

In [ ]:
reviews_train, reviews_test, y_train, y_test = train_test_split(
    reviews, categorias, test_size=0.3, random_state=RANDOM_STATE,
    stratify=categorias
)

print(f'Train : {len(reviews_train)} reseñas')
print(f'Test  : {len(reviews_test)} reseñas')

## 6 · Embeddings y entrenamiento

El modelo `paraphrase-multilingual-MiniLM-L12-v2` ya está pre-entrenado:  
no necesita `fit()` — se llama `encode()` por separado en train y test.  
La regresión logística recibe `multi_class='ovr'` — **sin cambios respecto a sesion_4**.

In [ ]:
# Cargar el modelo multilingüe pre-entrenado
encoder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Codificar los textos como vectores densos de 384 dimensiones
# No existe fit() — el modelo ya conoce el lenguaje
X_train = encoder.encode(reviews_train, show_progress_bar=True)
X_test  = encoder.encode(reviews_test,  show_progress_bar=True)

# El mismo clasificador OvR — solo cambia la representación de entrada
modelo = LogisticRegression(multi_class='ovr', random_state=RANDOM_STATE)
modelo.fit(X_train, y_train)

print('Modelo entrenado.')
print(f'Clases aprendidas : {modelo.classes_.tolist()}')
print(f'Dimensión embeddings: {X_train.shape[1]}')  # → 384

## 7 · Evaluación con sklearn

`classification_report` muestra automáticamente precision, recall y F1 **por clase** y el promedio macro.

In [ ]:
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred))

## 8 · Métricas manuales por clase

Las mismas funciones de v1 aplicadas en un loop:  
para cada categoría, el resto de clases actúan como «negativo».

In [ ]:
print(f"{'Categoría':<15} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-' * 48)

f1_por_clase = []
for c in CLASES:
    p = precision(y_test, y_pred, pos_label=c)
    r = recall(y_test, y_pred, pos_label=c)
    f = f1(y_test, y_pred, pos_label=c)
    f1_por_clase.append(f)
    print(f'{c:<15} {p:>10.2f} {r:>10.2f} {f:>10.2f}')

print('-' * 48)
macro_f1 = sum(f1_por_clase) / len(f1_por_clase)
print(f"{'Macro F1':<15} {'':>10} {'':>10} {macro_f1:>10.2f}")
print()
print(f'Accuracy global: {accuracy(y_test, y_pred):.2f}')

## 9 · Matriz de confusión 5×5

**Cómo leerla:**
- La **diagonal** (peliculas↔peliculas, restaurantes↔restaurantes…) son los aciertos.
- Los valores **fuera de la diagonal** son los errores — la fila dice la clase real, la columna la predicha.
- Una columna con muchos valores fuera de diagonal indica que el modelo "confunde" otras clases con esa.

In [ ]:
confusion_matrix_multiclase(y_test, y_pred, CLASES)

## 10 · Predicción sobre textos nuevos

In [ ]:
nuevas_reviews = [
    'La actuación fue brillante y la historia muy emotiva',
    'El sushi estaba fresco y el servicio impecable',
    'El producto llegó en perfectas condiciones y funciona genial',
    'Tardaron semanas en responder y no solucionaron el problema',
    'Habitación limpia, cama cómoda y muy buena ubicación',
]

# encoder.encode() en lugar de vectorizador.transform()
preds = modelo.predict(encoder.encode(nuevas_reviews))

for texto, pred in zip(nuevas_reviews, preds):
    print(f'  [{pred.upper():<13}]  {texto}')